# STEP 3: Cosine Similarity Retrieval

**Recap and explanation of this step:**
After having completed step 1 and 2 by embedding all 20,000 prompts into vectors and storing them in both a ChromaDB and a Qdrant vector database, we are ready to handle the query part.

This section handles the query side of the user, and given a natural language search query we will:
1. Embed the query using the **same model** used in Step 1 (`all-MiniLM-L6-v2`)
2. Search both ChromaDB and Qdrant using **cosine similarity** to find the closest matching prompts
3. Compare the two databases in terms of **latency and result quality** to determine the best one
4. Return ranked results (that will later be passed to the reranker in Step 4)


We run this on **both Config A** (content only) and **Config B** (title + category + subcategory + tags + content) so the reranker team can compare both.

## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers chromadb qdrant-client

## 2. Load the embedding model

We must use the **exact same model** that Step 1 used to embed the prompts.
Clearly, this is because if we used a different model, the query vector would live in a different "space" and cosine similarity would be meaningless.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # same as Step 1
embedding_model = SentenceTransformer(MODEL_NAME)

print(f"Model loaded: {MODEL_NAME}")

## 3. Connect to ChromaDB and Qdrant

Step 2 created two collections in ChromaDB (`config_a`, `config_b`) and two in Qdrant (`config_a_qdrant`, `config_b_qdrant`). We connect to both databases here.

In [ ]:
import chromadb
from qdrant_client import QdrantClient

# ChromaDB
chromaDB = chromadb.PersistentClient(path="/content")
config_a_chromadb = chromaDB.get_collection(name="config_a")
config_b_chromadb = chromaDB.get_collection(name="config_b")

# Qdrant
qdrant = QdrantClient(path="/content")

print(f"ChromaDB Config A Loading Check: {config_a_chromadb.count()} prompts")
print(f"ChromaDB Config B Loading Check: {config_b_chromadb.count()} prompts")
print(f"Qdrant Config A Loading Check: {qdrant.get_collection('config_a_qdrant').points_count} prompts")
print(f"Qdrant Config B Loading Check: {qdrant.get_collection('config_b_qdrant').points_count} prompts")

## 4. Retrieval functions

We implement one retrieval function for ChromaDB and one for Qdrant. Both:
- Take a natural language query and convert it into a vector
- Search the vector database using cosine similarity
- Return results sorted from most to least relevant, each with a similarity score and full metadata (including `likes`, `upvotes`, `difficulty` for the reranker)

In [ ]:
import numpy as np
import time

def retrieve_chromadb(query: str, collection, top_k: int = 20):
    # Embed the query
    query_vector = embedding_model.encode(query, convert_to_numpy=True).tolist()

    # Searching ChromaDB
    start = time.time()
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    latency = time.time() - start

    # ChromaDB returns lists-of-lists (one per query), so we take index [0]
    ids       = results["ids"][0]
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    # ChromaDB distance = 1 - cosine_similarity, so we convert back
    formatted = []
    for rank, (pid, doc, meta, dist) in enumerate(zip(ids, documents, metadatas, distances), start=1):
        formatted.append({
            "rank":        rank,
            "id":          pid,
            "similarity":  round(1 - dist, 4),
            "distance":    round(dist, 4),
            "title":       meta.get("title", ""),
            "category":    meta.get("category", ""),
            "subcategory": meta.get("subcategory", ""),
            "tags":        meta.get("tags", ""),
            "difficulty":  meta.get("difficulty", ""),
            "likes":       meta.get("likes", 0),
            "upvotes":     meta.get("upvotes", 0),
            "content":     doc,
        })

    return formatted, latency


def retrieve_qdrant(query: str, collection_name: str, top_k: int = 20):
    # Embed the query
    query_vector = embedding_model.encode(query, convert_to_numpy=True).tolist()

    # Searching Qdrant 
    start = time.time()
    results = qdrant.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        with_payload=True
    )
    latency = time.time() - start

    formatted = []
    for rank, point in enumerate(results.points, start=1):
        payload = point.payload or {}
        formatted.append({
            "rank":        rank,
            "id":          str(point.id),
            "similarity":  round(point.score, 4),
            "title":       payload.get("title", ""),
            "category":    payload.get("category", ""),
            "subcategory": payload.get("subcategory", ""),
            "tags":        payload.get("tags", ""),
            "difficulty":  payload.get("difficulty", ""),
            "likes":       payload.get("likes", 0),
            "upvotes":     payload.get("upvotes", 0),
            "content":     payload.get("content", ""),
        })

    return formatted, latency


def print_results(results, query, label):
    print(f"\n{'='*70}")
    print(f"Query: \"{query}\"")
    print(f"{'='*70}")
    for r in results:
        print(f"\n  #{r['rank']}  [{r['id']}]  similarity={r['similarity']}")
        print(f"  Title      : {r['title']}")
        print(f"  Category   : {r['category']} > {r['subcategory']}")
        print(f"  Tags       : {r['tags']}")
        print(f"  Difficulty : {r['difficulty']} | Likes: {r['likes']} | Upvotes: {r['upvotes']}")
        print(f"  Content    : {r['content'][:180]}...")
        print(f"  {'-'*66}")

## 5. ChromaDB vs Qdrant — Performance Comparison

Step 2 built both databases and here we measure which one is faster and/or returns better results. We do that here by running the same queries on both and comparing latency and top results.

**Note:** Given the dataset size of 20,000 prompts, both ChromaDB and Qdrant operate well within their capabilities. However, we know that since Qdrant's production-grade performance advantages are well documented in the literature, for this reason we expect that at this scale our results might not be significantly informative. Finally, we decided to include it for completeness and to provide a documented, empirical justification for our final database choice within this specific pipeline.

In [ ]:
benchmark_queries = [
    "write Python code to scrape a website",
    "summarize a legal contract",
    "create a marketing post for Instagram",
    "explain machine learning to a beginner",
    "generate a cold outreach email for sales",
]

chroma_latencies = []
qdrant_latencies = []
agreement_count = 0

print("ChromaDB vs Qdrant — Latency and Top Result Comparison (Config B)\n")
print(f"{'Query':<45} {'Chroma (ms)':>12} {'Qdrant (ms)':>12} {'Same top result?':>18}")
print("-" * 90)

for query in benchmark_queries:
    res_chroma, lat_chroma = retrieve_chromadb(query, config_b_chromadb, top_k=20)
    res_qdrant, lat_qdrant = retrieve_qdrant(query, "config_b_qdrant", top_k=20)

    chroma_latencies.append(lat_chroma)
    qdrant_latencies.append(lat_qdrant)

    same = res_chroma[0]["id"] == res_qdrant[0]["id"]
    if same:
        agreement_count += 1

    print(f"{query:<45} {lat_chroma*1000:>11.1f}ms {lat_qdrant*1000:>11.1f}ms {'YES' if same else 'NO':>18}")

print("-" * 90)
print(f"{'AVERAGE':<45} {sum(chroma_latencies)/len(chroma_latencies)*1000:>11.1f}ms {sum(qdrant_latencies)/len(qdrant_latencies)*1000:>11.1f}ms")
print(f"\nTop result agreement: {agreement_count}/{len(benchmark_queries)} queries")
print(f"\nWinner: {'ChromaDB' if sum(chroma_latencies) < sum(qdrant_latencies) else 'Qdrant'} is faster on average")

## 6. Running a search

Change `QUERY` to whatever you want to search for. We use the winning database from the comparison above.

In [ ]:
QUERY = "help me write a professional email to a client"
TOP_K = 20

results_a, _ = retrieve_chromadb(QUERY, config_a_chromadb, top_k=TOP_K)
results_b, _ = retrieve_chromadb(QUERY, config_b_chromadb, top_k=TOP_K)

print_results(results_a[:5], QUERY, "Config A (content only)")
print_results(results_b[:5], QUERY, "Config B (enriched: title + category + tags + content)")

## 7. A/B comparison across multiple queries

We test several queries and compare whether Config A or Config B returns better top results.
This is the A/B comparison the challenge evaluation requires.

In [ ]:
test_queries = [
    "write Python code to scrape a website",
    "summarize a legal contract",
    "create a marketing post for Instagram",
    "explain machine learning to a beginner",
    "generate a cold outreach email for sales",
]

print("Running A/B comparison across test queries...\n")

for query in test_queries:
    res_a, _ = retrieve_chromadb(query, config_a_chromadb, top_k=5)
    res_b, _ = retrieve_chromadb(query, config_b_chromadb, top_k=5)

    ids_a = [r["id"] for r in res_a]
    ids_b = [r["id"] for r in res_b]
    overlap = len(set(ids_a) & set(ids_b))

    print(f"\nQUERY: \"{query}\"")
    for i in range(5):
        match = "✓" if res_a[i]["id"] == res_b[i]["id"] else "✗"
        print(f"  #{i+1} {match}  A: [{res_a[i]['id']}] sim={res_a[i]['similarity']} | {res_a[i]['title'][:40]}")
        print(f"       B: [{res_b[i]['id']}] sim={res_b[i]['similarity']} | {res_b[i]['title'][:40]}")
    print(f"  → Shared prompts in top 5: {overlap}/5")

## 8. Evaluate retrieval quality — Precision@K and MRR

To measure quality we picked a small set of queries where we manually mark which results are truly relevant.

- **Precision@K**: of the top K results, what fraction are actually relevant?
- **MRR (Mean Reciprocal Rank)**: on average, how high up does the first relevant result appear? (1.0 = always #1, 0.5 = first relevant is at #2, etc.)

In [ ]:
GROUND_TRUTH = {
    "create a social media marketing campaign": [
        "pk_02364",  # Social media visual campaign
        "pk_00017",  # Social Media Content Calendar
        "pk_14568",  # Social Media Content Calendar Generator
        "pk_16478",  # Viral Social Media Post Formulas
        "pk_01267",  # Create Hashtag Strategy Plan
    ],
    "write a response to a negative customer review": [
        "pk_06217",  # Negative Review Response Templates (ticket-responses)
        "pk_04336",  # Negative Review Response Templates (ticket-responses)
        "pk_19170",  # Negative Review Response Templates (marketing)
        "pk_11972",  # Create negative review response strategy
        "pk_07091",  # Complaint Acknowledgment Response
    ],
    "write a SQL query to analyze database data": [
        "pk_19178",  # SQL Customer Segmentation
        "pk_18782",  # SQL Query for Monthly Revenue
        "pk_00054",  # Design E-commerce Database
        "pk_00632",  # Optimize slow database queries
        "pk_17577",  # Optimize Slow Database Queries
    ],
}


def precision_at_k(results, relevant_ids, k):
    top_k_ids = [r["id"] for r in results[:k]]
    hits = sum(1 for pid in top_k_ids if pid in relevant_ids)
    return hits / k


def reciprocal_rank(results, relevant_ids):
    for i, r in enumerate(results, start=1):
        if r["id"] in relevant_ids:
            return 1.0 / i
    return 0.0


def evaluate(collection, ground_truth, top_k=10, label=""):
    p_scores, rr_scores = [], []
    for query, relevant_ids in ground_truth.items():
        if not relevant_ids:
            continue
        results, _ = retrieve_chromadb(query, collection, top_k=top_k)
        p = precision_at_k(results, set(relevant_ids), k=top_k)
        rr = reciprocal_rank(results, set(relevant_ids))
        p_scores.append(p)
        rr_scores.append(rr)
        print(f"  Query: '{query[:50]}' | P@{top_k}={p:.2f} | RR={rr:.2f}")

    if p_scores:
        print(f"\n  Config {label} — Mean P@{top_k}: {np.mean(p_scores):.3f} | MRR: {np.mean(rr_scores):.3f}")
    else:
        print(f"  Config {label} — No ground truth yet. Add relevant IDs to GROUND_TRUTH above.")


print("=== Config A ===")
evaluate(config_a_chromadb, GROUND_TRUTH, top_k=10, label="A")

print("\n=== Config B ===")
evaluate(config_b_chromadb, GROUND_TRUTH, top_k=10, label="B")

## 9. Export results for Step 4 (the reranker)

We package the top candidates for a given query into a clean format
that the reranker team (Step 4) can directly consume.
Each result includes `likes`, `upvotes`, and `difficulty` so the reranker can use them for metadata-aware scoring.

In [ ]:
import json

def get_candidates_for_reranker(query: str, top_k: int = 50):
    results_a, _ = retrieve_chromadb(query, config_a_chromadb, top_k=top_k)
    results_b, _ = retrieve_chromadb(query, config_b_chromadb, top_k=top_k)

    output = {
        "query": query,
        "top_k": top_k,
        "config_a_candidates": results_a,
        "config_b_candidates": results_b,
    }
    return output

QUERY = "help me write a professional email to a client"
candidates = get_candidates_for_reranker(QUERY, top_k=50)

with open("/content/candidates_for_reranker.json", "w") as f:
    json.dump(candidates, f, indent=2)

print(f"Exported {len(candidates['config_a_candidates'])} candidates (Config A)")
print(f"Exported {len(candidates['config_b_candidates'])} candidates (Config B)")
print("Saved to: /content/candidates_for_reranker.json")